# Edge pilot (all stages + unverified-item checks)

Step `edge-pilot`. Thin wrapper; logic lives in `parameters.edge.pilot`.


In [ ]:
import os, subprocess
REPO, URL, BRANCH = "/kaggle/working/repo", "https://github.com/saltypal/SemanticSchedulerEnd2End.git", "main"
if os.path.isdir(REPO + "/.git"):
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", "origin/" + BRANCH], check=True)
else:
    subprocess.run(f"git clone --depth 1 --branch {BRANCH} {URL} {REPO}", shell=True, check=True)
GIT_SHA = subprocess.check_output(f"git -C {REPO} rev-parse HEAD", shell=True, text=True).strip()
print("commit", GIT_SHA)

In [ ]:
%cd /kaggle/working/repo
!pip install -q -e . pytorch-lightning==1.9.5 torchmetrics==0.11.4 transformers==4.46.3 scipy omegaconf pytorch-msssim==1.0.0 lpips==0.1.4 PyYAML einops

In [ ]:
import time; T0 = time.time()

In [ ]:
!python -u -m parameters.edge.pilot --work /kaggle/working/pilot --per-source 20

## Collect report

Copies the report and per-check JSON to /kaggle/working and prints it.


In [ ]:
import json, shutil
from pathlib import Path
W, OUT = Path("/kaggle/working/pilot"), Path("/kaggle/working")
for name in ("PILOT_REPORT.md", "pilot_report.json"):
    if (W / name).exists():
        shutil.copyfile(W / name, OUT / name)
if (W / "checks").is_dir():
    shutil.copytree(W / "checks", OUT / "checks", dirs_exist_ok=True)
print((OUT / "PILOT_REPORT.md").read_text() if (OUT / "PILOT_REPORT.md").exists() else "NO PILOT_REPORT.md PRODUCED")

In [ ]:
import datetime, json, os, platform, subprocess, time
STEP, ARGS = "edge-pilot", ['--work', '/kaggle/working/pilot', '--per-source', '20']
def _gpu():
    try:
        return subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
    except Exception:
        return None
try:
    import torch; torch_version = torch.__version__
except Exception:
    torch_version = None
out = "/kaggle/working/runs/" + STEP
os.makedirs(out, exist_ok=True)
manifest = {"step": STEP, "entrypoint": "parameters.edge.pilot", "args": ARGS, "git_sha": GIT_SHA,
            "utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "python": platform.python_version(),
            "torch": torch_version, "gpu": _gpu(), "wall_time_s": time.time() - T0}
json.dump(manifest, open(out + "/run_manifest.json", "w"), indent=2)
print(manifest)